# Interpretación del Modelo — Coeficientes y SHAP Values

## Objetivo de este notebook

Explicar **qué variables impulsan el precio predicho** y en qué dirección,
usando dos enfoques complementarios sobre el modelo seleccionado (Regresión Lineal Baseline):

1. **Coeficientes de Regresión** — impacto directo de cada variable en `SalePrice_log`
   - Útil para entender dirección del efecto (positivo/negativo)
   - Limitación: distorsionado por One-Hot Encoding (fragmenta variables categóricas)

2. **SHAP Values** (SHapley Additive exPlanations) — contribución individual por observación
   - Basado en teoría de juegos de Shapley: distribución justa del crédito entre features
   - Corrige la distorsión del One-Hot Encoding
   - Permite explicaciones a nivel de propiedad individual

**Input:** `data/train_features.csv` + modelo Baseline Lineal reentrenado

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================
import shap  # SHAP Values — interpretabilidad modelo-agnóstica basada en teoría de juegos

import pandas as pd
import numpy as np
from pathlib import Path

# Modelos y evaluación
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import Ridge, RidgeCV, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Crea la carpeta images/ en la raíz del proyecto si no existe
Path('../../images').mkdir(exist_ok=True)

In [5]:
data_path = Path('../../data/train_features.csv')
# keep_default_na=False desactiva la interpretación automática de nulos
# na_values=[] lista vacía — ningún valor adicional se interpreta como nulo
df = pd.read_csv(data_path, keep_default_na=False, na_values=[''])

print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Nulos totales: {df.isnull().sum().sum()}')
df.columns

Filas: 1460
Columnas: 47
Nulos totales: 0


Index(['OverallQual', 'GrLivArea', 'GarageArea', 'TotalBsmtSF', 'FullBath',
       'MasVnrArea', 'Fireplaces', 'BsmtFinSF1', 'LotFrontage', 'WoodDeckSF',
       '2ndFlrSF', 'OpenPorchSF', 'HalfBath', 'property_age',
       'year_since_remod', 'ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC',
       'BsmtExposure', 'BsmtFinType1', 'GarageFinish', 'PavedDrive',
       'LotShape', 'Foundation_CBlock', 'Foundation_PConc', 'Foundation_Slab',
       'Foundation_Stone', 'Foundation_Wood', 'GarageType_Attchd',
       'GarageType_Basment', 'GarageType_BuiltIn', 'GarageType_CarPort',
       'GarageType_Detchd', 'GarageType_None', 'MSZoning_FV', 'MSZoning_RH',
       'MSZoning_RL', 'MSZoning_RM', 'SaleCondition_AdjLand',
       'SaleCondition_Alloca', 'SaleCondition_Family', 'SaleCondition_Normal',
       'SaleCondition_Partial', 'CentralAir', 'Neighborhood', 'SalePrice_log'],
      dtype='object')

In [ ]:
X = df.drop('SalePrice_log', axis=1)
y = df['SalePrice_log']

print(f'Shape X: {X.shape}')
print(f'Shape Y: {y.shape}')

# Mismo split que notebooks 04 y 05 — random_state=42 garantiza comparación válida
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training Data Set: {x_train.shape[0]} filas")
print(f"Test Data Set: {x_test.shape[0]} filas")

# Reentrenamos el modelo seleccionado (Baseline Lineal) para calcular los SHAP Values
# Es el mismo modelo que en el Notebook 04 — resultados idénticos garantizados
model_log = LinearRegression()
model_log.fit(x_train, y_train)

y_pred_log = model_log.predict(x_test)

# Métricas — deben coincidir exactamente con Notebook 04
mae_log  = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log   = r2_score(y_test, y_pred_log)

print(f"MAE (log): {mae_log:.4f}")
print(f"RMSE (log): {rmse_log:.4f}")
print(f"R² (log): {r2_log:.4f}")

y_pred_usd = np.exp(y_pred_log)
y_test_usd = np.exp(y_test)

mae_usd  = mean_absolute_error(y_test_usd, y_pred_usd)
rmse_usd = np.sqrt(mean_squared_error(y_test_usd, y_pred_usd))
r2_usd   = r2_score(y_test_usd, y_pred_usd)

print(f"MAE (USD): ${mae_usd:,.0f}")
print(f"RMSE (USD): ${rmse_usd:,.0f}")
print(f"R² (USD): {r2_usd:.4f}")

## Feature Importance - Coeficientes Regresión Lineal

### ¿Qué se hace?
Se extraen los coeficientes del modelo de regresión lineal para identificar
qué variables tienen mayor impacto en la predicción del precio de venta.
Los coeficientes se ordenan por valor absoluto para medir el impacto
independientemente de su dirección (positiva o negativa).

### ¿Por qué valor absoluto?
Un coeficiente positivo indica que la variable aumenta el precio de venta.
Un coeficiente negativo indica que la variable reduce el precio de venta.
El valor absoluto permite comparar el impacto sin importar la dirección.

### Resultados obtenidos
Las variables con mayor coeficiente absoluto corresponden en su mayoría
a variables de MSZoning y SaleCondition, seguidas de CentralAir,
GarageType y OverallQual.

### Insight critico - Limitacion del analisis por coeficientes
Las variables MSZoning y SaleCondition dominan el top 15 debido al
One-Hot Encoding aplicado en la Fase 3. Al fragmentar una sola variable
categórica en multiples columnas binarias, el modelo asigna coeficientes
individuales a cada categoria, lo que distorsiona la comparacion global.

Por ejemplo:
- MSZoning original    → 1 variable con F-statistic = 43.84 (EDA Fase 1)
- MSZoning One-Hot     → 4 columnas en el top 15 del ranking

Esto desplaza variables como OverallQual (correlacion 0.79 con SalePrice)
que deberia estar entre las mas importantes segun el EDA.

### Conclusion
Los coeficientes son utiles para entender la direccion del impacto de cada
variable, pero no son confiables para comparar importancia global cuando
existe One-Hot Encoding. Esto justifica el uso de SHAP Values en el
siguiente bloque, que calcula el impacto real de cada variable original
independientemente de su codificacion.

In [ ]:
# Extraemos coeficientes del modelo y calculamos valor absoluto para medir impacto
# independientemente de la dirección (positiva o negativa)
coeficientes = pd.DataFrame({
    'Feature': X.columns,
    'Coeficiente': model_log.coef_
})

coeficientes['abs_coef'] = coeficientes['Coeficiente'].abs()
coeficientes = coeficientes.sort_values(by='abs_coef', ascending=False)

top_15 = coeficientes.head(15)

# Barras horizontales — color: positivo sube precio, negativo lo reduce
plt.figure(figsize=(10, 6))
plt.barh(top_15['Feature'], top_15['Coeficiente'],
         color=['steelblue' if c > 0 else 'tomato' for c in top_15['Coeficiente']])
plt.xlabel('Valor del Coeficiente')
plt.title('Top 15 Características por Valor Absoluto del Coeficiente')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../../images/06_coeficientes_top15.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## FASE 6 - INTERPRETACION: SHAP VALUES

### ¿Qué son los SHAP Values?
SHAP (SHapley Additive exPlanations) es un método que explica la prediccion
de cualquier modelo calculando la contribucion individual de cada variable
sobre la diferencia entre el precio predicho y el precio promedio del dataset.

A diferencia de los coeficientes de regresion lineal, SHAP calcula el impacto
a nivel de observacion individual — es decir, por cada propiedad especifica.

**Lectura del grafico:**
- Eje X: SHAP value — impacto de la variable en la prediccion (escala log)
  - Derecha (SHAP > 0) → la variable aumenta el precio predicho
  - Izquierda (SHAP < 0) → la variable reduce el precio predicho
- Color: valor de la feature
  - Rojo → valor alto de la variable
  - Azul → valor bajo de la variable

**Nota:** Los SHAP values estan en escala logaritmica. Valores mayores a 0.1
representan impactos economicamente significativos en el precio de venta.


In [ ]:
# LinearExplainer calcula SHAP values aprovechando la linealidad del modelo
# Es más eficiente que TreeExplainer o KernelExplainer para regresión lineal
explainer = shap.LinearExplainer(
    model_log,   # modelo entrenado
    x_train      # datos de referencia para calcular la contribución marginal de cada feature
)

# Calculamos SHAP values sobre el test set — un valor por feature por observación
shap_values = explainer(x_test)

# Summary plot: cada punto es una observación, el eje X muestra el impacto en la predicción
# show=False permite guardar la figura antes de mostrarla
shap.summary_plot(
    shap_values,
    x_test,
    max_display=15,
    show=False
)
plt.savefig('../../images/06_shap_summary_plot.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

### Conclusiones de Negocio

**1. El area habitable es el driver principal del precio**
GrLivArea domina el modelo con SHAP values de hasta +0.6 en log-escala.
Casas con mayor area habitable tienen un impacto positivo considerable
en el precio — es la variable mas accionable para un vendedor o comprador.

**2. La ubicacion importa tanto como la estructura**
MSZoning_RL, Neighborhood y MSZoning_RM aparecen en el top 6.
Las zonas residenciales de baja densidad tienen impacto positivo consistente,
confirmando que en Ames la zonificacion es un proxy de exclusividad y tamaño.

**3. La calidad supera a la antiguedad**
OverallQual en el top 3 vs property_age en el puesto 11 indica que el mercado
de Ames valora mas la calidad de construccion que la modernidad de la propiedad.
Una casa antigua bien construida supera en precio a una nueva de menor calidad.

**4. El garaje tiene valor propio**
GarageType_Attchd y GarageArea en el top 10 confirman que el tipo y tamaño
del garaje son componentes relevantes del valor — no un accesorio secundario.

**5. La remodelacion tiene impacto limitado**
year_since_remod aparece en el puesto 13 con impacto modesto.
Remodelar una propiedad antes de vender no garantiza un aumento
significativo en el precio predicho por el modelo.

**6. La edad de la propiedad tiene efecto contraintuitivo**
property_age muestra que casas mas antiguas tienden a tener mayor precio
predicho. Esto no implica que envejecer una casa suba su valor — sino que
en Ames las propiedades antiguas estan correlacionadas con mayor tamaño
y mejores ubicaciones historicas consolidadas.